In [2]:
import re
import math
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

nltk.download('stopwords', quiet=True)
docs = {
    1: "Information retrieval is the activity of obtaining relevant information from a collection.",
    2: "Web search engines use crawlers to collect web pages for indexing.",
    3: "An inverted index maps each term to the list of documents containing that term.",
    4: "Ranking of retrieved documents depends on term frequency and inverse document frequency.",
    5: "Text classification and clustering are important tasks in information retrieval.",
}

stop = set(stopwords.words('english'))
ps = PorterStemmer()
def preprocess(text):
    toks = re.findall(r'[a-z]+', text.lower())
    return [ps.stem(t) for t in toks if t not in stop]

index = defaultdict(dict)
doc_len = {}
for d, text in docs.items():
    terms = preprocess(text)
    doc_len[d] = len(terms)
    for t in terms:
        index[t][d] = index[t].get(d, 0) + 1

print("Vocabulary size:", len(index))
print("\nSample postings:")
for t in ['inform', 'retriev', 'index', 'document', 'term']:
    print(f" {t:<10} df={len(index[t])} postings={dict(sorted(index[t].items()))}")
ALL = set(docs)
def AND(a, b): return a & b
def OR(a, b): return a | b
def NOT(a): return ALL - a
def posting(term):
    return set(index.get(ps.stem(term.lower()), {}).keys())
print("\nBoolean queries:")
print(" information AND retrieval ->", sorted(AND(posting('information'), posting('retrieval'))))
print(" term OR crawlers ->", sorted(OR(posting('term'), posting('crawlers'))))
print(" information AND NOT classification ->",
      sorted(AND(posting('information'), NOT(posting('classification')))))
N = len(docs)
def rank(query, k=3):
    scores = defaultdict(float)
    for t in preprocess(query):
        if t in index:
            idf = math.log10(N / len(index[t]))
            for d, tf in index[t].items():
                scores[d] += (1 + math.log10(tf)) * idf
    return sorted(scores.items(), key=lambda x: -x[1])[:k]

q = "inverted index for document retrieval"
print(f"\nRanked results for query: '{q}'")
for d, s in rank(q):
    print(f" Doc {d} score={s:.3f} -> {docs[d]}")

Vocabulary size: 28

Sample postings:
 inform     df=2 postings={1: 2, 5: 1}
 retriev    df=3 postings={1: 1, 4: 1, 5: 1}
 index      df=2 postings={2: 1, 3: 1}
 document   df=2 postings={3: 1, 4: 2}
 term       df=2 postings={3: 2, 4: 1}

Boolean queries:
 information AND retrieval -> [1, 5]
 term OR crawlers -> [2, 3, 4]
 information AND NOT classification -> [1]

Ranked results for query: 'inverted index for document retrieval'
 Doc 3 score=1.495 -> An inverted index maps each term to the list of documents containing that term.
 Doc 4 score=0.740 -> Ranking of retrieved documents depends on term frequency and inverse document frequency.
 Doc 2 score=0.398 -> Web search engines use crawlers to collect web pages for indexing.


In [ ]:
Name- Devesh Mulani
Roll number- 4141